Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 1
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43788, 12, 6)
Dimensiones de Y: (43788, 1)


In [16]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30651, 12, 6)
Las dimensiones de testX son:  (8801, 12, 6)
Las dimensiones de valX son:  (4336, 12, 6)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30651, 1)
Las dimensiones de testY son:  (8801, 1)
Las dimensiones de valY son:  (4336, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

240/240 - 15s - 61ms/step - ia: 0.3667 - loss: 1.6778 - mae: 0.9137 - rmse: 1.2839 - smape: 1.2401 - val_ia: 0.2625 - val_loss: 1.0516 - val_mae: 0.6821 - val_rmse: 0.8585 - val_smape: 1.1436

Epoch 2/128                                           

240/240 - 4s - 18ms/step - ia: 0.2769 - loss: 1.1841 - mae: 0.7857 - rmse: 1.0817 - smape: 1.4268 - val_ia: 0.2438 - val_loss: 0.9651 - val_mae: 0.7147 - val_rmse: 0.8589 - val_smape: 1.6103

Epoch 3/128                                           

240/240 - 4s - 17ms/step - ia: 0.2388 - loss: 1.1183 - mae: 0.7832 - rmse: 1.0510 - smape: 1.5025 - val_ia: 0.2491 - val_loss: 0.9631 - val_mae: 0.7286 - val_rmse: 0.8684 - val_smape: 1.8382

Epoch 4/128                                           

240/240 - 4s - 16ms/step - ia: 0.2229 - loss: 1.0826 - mae: 0.7743 - rmse: 1.0346 - smape: 1.5284 - val_ia: 0.2505 - val_loss: 0.9596 - val_mae: 0.7274 - val_rmse: 0.8670 - val_smape: 1.8442

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                        

1916/1916 - 45s - 24ms/step - ia: 0.8544 - loss: 0.1037 - mae: 0.1925 - rmse: 0.2817 - smape: 0.4592 - val_ia: 0.6359 - val_loss: 0.0935 - val_mae: 0.1792 - val_rmse: 0.2375 - val_smape: 0.3846

Epoch 2/128                                                                        

1916/1916 - 48s - 25ms/step - ia: 0.8890 - loss: 0.0669 - mae: 0.1512 - rmse: 0.2288 - smape: 0.3771 - val_ia: 0.6664 - val_loss: 0.0577 - val_mae: 0.1413 - val_rmse: 0.1927 - val_smape: 0.3683

Epoch 3/128                                                                        

1916/1916 - 37s - 20ms/step - ia: 0.8920 - loss: 0.0661 - mae: 0.1480 - rmse: 0.2267 - smape: 0.3681 - val_ia: 0.6839 - val_loss: 0.0535 - val_mae: 0.1321 - val_rmse: 0.1835 - val_smape: 0.3543

Epoch 4/128                                                                        

1916/1916 - 52s - 27ms/step - ia: 0.8925 - loss: 0.0643 - mae: 0.1467 - rms

...

Epoch 1/128                                                                          

1916/1916 - 42s - 22ms/step - ia: 0.4631 - loss: 0.7617 - mae: 0.6332 - rmse: 0.8214 - smape: 1.1812 - val_ia: 0.3988 - val_loss: 0.2005 - val_mae: 0.3095 - val_rmse: 0.3713 - val_smape: 0.6991

Epoch 2/128                                                                          

1916/1916 - 26s - 14ms/step - ia: 0.7195 - loss: 0.2699 - mae: 0.3713 - rmse: 0.4944 - smape: 0.7606 - val_ia: 0.4725 - val_loss: 0.1404 - val_mae: 0.2426 - val_rmse: 0.3029 - val_smape: 0.5625

Epoch 3/128                                                                          

1916/1916 - 26s - 14ms/step - ia: 0.7614 - loss: 0.2026 - mae: 0.3155 - rmse: 0.4257 - smape: 0.6753 - val_ia: 0.5105 - val_loss: 0.1130 - val_mae: 0.2178 - val_rmse: 0.2764 - val_smape: 0.5207

Epoch 4/128                                                                          

1916/1916 - 27s - 14ms/step - ia: 0.7806 - loss: 0.1739 - mae: 0.28

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

3832/3832 - 42s - 11ms/step - ia: 0.3221 - loss: 0.9890 - mae: 0.7088 - rmse: 0.9118 - smape: 1.4982 - val_ia: 0.2012 - val_loss: 0.8420 - val_mae: 0.6674 - val_rmse: 0.7038 - val_smape: 1.5239

Epoch 2/128                                                                            

3832/3832 - 33s - 9ms/step - ia: 0.3325 - loss: 0.9530 - mae: 0.6980 - rmse: 0.8968 - smape: 1.4849 - val_ia: 0.2027 - val_loss: 0.8261 - val_mae: 0.6602 - val_rmse: 0.6965 - val_smape: 1.5026

Epoch 3/128                                                                            

3832/3832 - 31s - 8ms/step - ia: 0.3366 - loss: 0.9385 - mae: 0.6936 - rmse: 0.8898 - smape: 1.4737 - val_ia: 0.2042 - val_loss: 0.8111 - val_mae: 0.6535 - val_rmse: 0.6896 - val_smape: 1.4828

Epoch 4/128                                                                            

3832/3832 - 35s - 9ms/step - ia: 0.3422 - loss: 0.9269 - mae:

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

240/240 - 7s - 31ms/step - ia: 0.4495 - loss: 0.7126 - mae: 0.6185 - rmse: 0.8279 - smape: 1.2121 - val_ia: 0.5912 - val_loss: 0.3045 - val_mae: 0.4018 - val_rmse: 0.4997 - val_smape: 0.8877

Epoch 2/128                                                                            

240/240 - 3s - 13ms/step - ia: 0.7165 - loss: 0.2833 - mae: 0.3904 - rmse: 0.5265 - smape: 0.8033 - val_ia: 0.7619 - val_loss: 0.1365 - val_mae: 0.2449 - val_rmse: 0.3367 - val_smape: 0.5639

Epoch 3/128                                                                            

240/240 - 3s - 13ms/step - ia: 0.7784 - loss: 0.1938 - mae: 0.3178 - rmse: 0.4361 - smape: 0.6843 - val_ia: 0.7974 - val_loss: 0.1050 - val_mae: 0.2047 - val_rmse: 0.2957 - val_smape: 0.4978

Epoch 4/128                                                                            

240/240 - 3s - 13ms/step - ia: 0.7997 - loss: 0.1647 - mae: 0.2874 -

In [23]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
